## 1. Dependency installation:
In order to use reactive Rust Jupyter Notebook, follow instruction [here](https://github.com/evcxr/evcxr/blob/main/evcxr_jupyter/README.md) to install Rust Kernel for Jupyter Notebook.

The compilation takes quite some time for the first time. This is a nature of the programming language itself. Be patient.

In [2]:
// Load dependecies
:dep ndarray = { version = "0.16.1" }
:dep ndarray-rand = { version = "0.15.0" }
:dep polars = { version = "0.49.1", features = ["describe", "lazy", "ndarray"] }
:dep plotters = { version = "0.3.5", features = ["evcxr", "all_series", "all_elements"] }
:dep rayon = { version = "1.10.0" }

In [3]:
// Show dependecies
:show_deps

ndarray = { version = "0.16.1" }
ndarray-rand = { version = "0.15.0" }
plotters = { version = "0.3.5", features = ["evcxr", "all_series", "all_elements"] }
polars = { version = "0.49.1", features = ["describe", "lazy", "ndarray"] }
rayon = { version = "1.10.0" }


## 2. Model S/M introduction

### 2.1 Model description:
The reference is based on [Basic term - model S](https://lifelib.io/libraries/basiclife/BasicTerm_M.html)

### 2.2 Product feature:
- Level-premium plain term product
- No surrender value.
                                                 
### 2.3 Assumptions:
- Mortality rates
- Lapse rates
- Discount rate
- Expense
- Inflation
- Commission rates.

### 2.4 Model:
The most basic cashflow model:
- Monthly step
- New business model
- Projects insurance cashflows of a sample model point: premiums, claims, expenses and commissions.
- The present values of the cashflows are also calculated.
- The premium amount for each individual model point is calculated as the net premium with loadings, where the net premium is calculated from the present value of the claims.

In our demonstration:
- Term has 3 options 10, 15, 20 years
- Commission is 100% premium in the first year

## 3. Modules
Each cell below represent a seperate module that is structured in order to perform the task more conviniently.

In Jupyter note book, they are all loaded into one notebook.

### 3.0 Load all libraries
Since the notebook basically store all functions in one plain space, we first load all the libraries.

In [4]:
// Load all libraries
use ndarray::prelude::*;
use ndarray_rand::RandomExt;
use ndarray_rand::rand::SeedableRng;
use ndarray_rand::rand::rngs::StdRng;
use ndarray_rand::rand_distr::Uniform;

use polars::prelude::*;

use rayon::prelude::*;

use std::path::PathBuf;
use std::fs::File;

### 3.1 Model point module:
These functions should be placed in the same module (eg: model_points.rs)

In [5]:
//---------------------------------------------------------------------------------------------------------
// STRUCTS
//---------------------------------------------------------------------------------------------------------
pub struct ModelPoint {
    pub id: i32,
    pub entry_age: i32,
    pub gender: String,
    pub term: i32,
    pub policy_count: f64,
    pub sum_insured: f64,
}

pub fn generate_model_points(mp_size: usize, seed: usize) -> PolarsResult<DataFrame> {
    // Get seed for random number generation
    let mut rng = StdRng::seed_from_u64(seed as u64);

    // Issue Age (Integer): Random 20 - 59 year old
    let entry_age = Array1::random_using(mp_size, Uniform::new(20, 60), &mut rng); // 60 is exclusive, so range is 20-59

    // Gender (String): Random "M" and "F"
    let gender_binary = Array1::random_using(mp_size, Uniform::new(0, 2), &mut rng); // 0 or 1
    let gender: Vec<&str> = gender_binary
        .iter()
        .map(|&x| if x == 0 { "M" } else { "F" }) // map 0 to "M" and 1 to "F"
        .collect();

    // Policy term (Integer): Random 10, 15 or 20
    let term = (Array1::random_using(mp_size, Uniform::new(2, 5), &mut rng)) * 5;

    // Policy count
    let policy_count = Array1::<f64>::ones(mp_size);

    // Sum insured (Float): Random values between 100,000 and 1,000,000 (multiple of 1000)
    let sum_insured =
        Array1::random_using(mp_size, Uniform::new(0.0f64, 1.0f64), &mut rng) // Random floats between 0 and 1
            .mapv(|x| (((900_000.0 * x + 100_000.0) / 1000.0).round() * 1000.0) as f64);

    // Create a DataFrame with the generated data
    let model_points_df = df![
        "id"  => (1..(mp_size+1) as i32).collect::<Vec<i32>>(),
        "entry_age" => entry_age.to_vec(),
        "gender" => gender,
        "term" => term.to_vec(),
        "policy_count" => policy_count.to_vec().into_iter().map(|x| x as f64).collect::<Vec<f64>>(),
        "sum_insured" => sum_insured.to_vec(),
    ]?;

    Ok(model_points_df)
}

pub fn convert_model_points_df_to_vector(df: &DataFrame) -> PolarsResult<Vec<ModelPoint>> {
    let id = df.column("id")?.i32()?;
    let entry_age = df.column("entry_age")?.i32()?;
    let gender = df.column("gender")?.str()?;
    let term = df.column("term")?.i32()?;
    let policy_count = df.column("policy_count")?.f64()?;
    let sum_insured = df.column("sum_insured")?.f64()?;

    let model_points = (0..df.height())
        .map(|i| ModelPoint {
            id: id.get(i).unwrap(),
            entry_age: entry_age.get(i).unwrap(),
            gender: gender.get(i).unwrap().to_string(),
            term: term.get(i).unwrap(),
            policy_count: policy_count.get(i).unwrap(),
            sum_insured: sum_insured.get(i).unwrap(),
        })
        .collect();

    Ok(model_points)
}

### 3.2 Assumption module

In [6]:
//---------------------------------------------------------------------------------------------------------
// STRUCTS
//---------------------------------------------------------------------------------------------------------
#[derive(Debug, Clone)]
pub struct AssumptionSet {
    pub mort: DataFrame,
    pub lapse: DataFrame,
    pub inf: DataFrame,
    pub acq: DataFrame,
    pub mtn: DataFrame,
    pub spot: DataFrame,
    pub load: DataFrame,
}

//---------------------------------------------------------------------------------------------------------
// PRIVATE
//---------------------------------------------------------------------------------------------------------
fn _get_assumption_df(
    file_path_str: &str,
    col_name: &str,
    new_col_name: &str,
) -> PolarsResult<DataFrame> {
    let lapse_file_path = PathBuf::from(file_path_str);

    CsvReadOptions::default()
        .try_into_reader_with_file_path(Some(lapse_file_path))?
        .finish()?
        .lazy()
        .select([col("year"), col(col_name).alias(new_col_name)]) // Select which column as a rate
        .collect()
}

//---------------------------------------------------------------------------------------------------------
// PUBLIC
//---------------------------------------------------------------------------------------------------------
// Lapse assumption
pub fn get_lapse_df(lapse_type: &str) -> PolarsResult<DataFrame> {
    _get_assumption_df("src/assumptions/lapse.csv", lapse_type, "lapse_rate")
}

// Inflation assumption
pub fn get_inf_rate_df(inf_type: &str) -> PolarsResult<DataFrame> {
    _get_assumption_df("src/assumptions/inflation.csv", inf_type, "inf_rate")
}

// Acquisition assumption
pub fn get_acq_exp_df(acq_exp_type: &str) -> PolarsResult<DataFrame> {
    _get_assumption_df(
        "src/assumptions/acq_exp.csv",
        acq_exp_type,
        "real_acq_exp_pp",
    )
}

// Maintenance assumption
pub fn get_mtn_exp_df(mtn_exp_type: &str) -> PolarsResult<DataFrame> {
    _get_assumption_df(
        "src/assumptions/mtn_exp.csv",
        mtn_exp_type,
        "real_mtn_exp_pp",
    )
}

pub fn get_spot_rate_df(spot_rate_type: &str) -> PolarsResult<DataFrame> {
    _get_assumption_df("src/assumptions/spot_rate.csv", spot_rate_type, "spot_rate")
}

pub fn get_load_rate_df(spot_rate_type: &str) -> PolarsResult<DataFrame> {
    _get_assumption_df("src/assumptions/load.csv", spot_rate_type, "load_rate")
}

// Mortality assumption: The schema is slightly different from other since it is based on gender
pub fn get_mort_df(mort_type: &str) -> PolarsResult<DataFrame> {
    let mort_file_path = PathBuf::from("src/assumptions/mort.csv");

    CsvReadOptions::default()
        .try_into_reader_with_file_path(Some(mort_file_path))?
        .finish()?
        .select([
            "age",
            &format!("{}_m", mort_type),
            &format!("{}_f", mort_type),
        ])
}

### 3.3 Projection module

In [ ]:
//---------------------------------------------------------------------------------------------------------
// STRUCTS
//---------------------------------------------------------------------------------------------------------
#[derive(Clone, Debug)]
pub struct RunSetup {
    pub description: String, // Optional description for the run
    pub model_points_df: DataFrame,
    pub assumptions: AssumptionSet,
}

#[derive(Clone, Debug)]
pub struct RunResult {
    pub run_setup: RunSetup,
    pub projected_df: DataFrame,
}

#[derive(Clone, Debug)]
pub struct MultipleRunResult {
    pub run_setups: Vec<RunSetup>,
    pub run_results: Vec<RunResult>,
}

impl MultipleRunResult {
    pub fn aggregate_projected_df(&self) -> PolarsResult<DataFrame> {
        let mut lfs: Vec<LazyFrame> = Vec::with_capacity(self.run_results.len());

        for (i, run_result) in self.run_results.iter().enumerate() {
            let run_id = i as i32;
            let run_description = run_result.run_setup.description.clone();

            let lf = run_result.projected_df.clone().lazy().with_columns(vec![
                lit(run_id).alias("run_id"),
                lit(run_description).alias("run_description"),
            ]);
            lfs.push(lf);
        }

        let lf = concat(lfs, Default::default())?;
        lf.collect()
    }
}

//---------------------------------------------------------------------------------------------------------
// PRIVATE
//---------------------------------------------------------------------------------------------------------
// Initialize projection lazyframe
fn __initialize_lf(
    id: i32,
    term: i32,
    entry_age: i32,
    sum_insured: f64,
) -> PolarsResult<LazyFrame> {
    let length = (term * 12 + 1) as usize; // Total months in the term

    let lf = df![
        "id" => vec![id; length],
        "term" => vec![term; length],
        "sum_insured" => vec![sum_insured; length],
        "t" => (0..= (length -1) as i32).collect::<Vec<i32>>(),
    ]
    .unwrap()
    .lazy()
    .with_column((col("t") / lit(12)).alias("duration"))
    .with_column((lit(entry_age) + col("duration")).alias("age"));

    Ok(lf.collect()?.lazy()) // To avoid nested lazyframe
}

// Map mortality assumption
fn __map_mort_assumption(
    lf: LazyFrame,
    mort_df: &DataFrame,
    gender: &str,
) -> PolarsResult<LazyFrame> {
    // Find the column name according to gender
    let suffix = format!("_{}", gender.to_lowercase());
    let mort_col_name = mort_df
        .get_column_names()
        .iter()
        .find(|&col| col.ends_with(&suffix))
        .ok_or_else(|| {
            polars::prelude::PolarsError::ComputeError(
                format!("Mortality column with suffix '{}' not found", suffix).into(),
            )
        })?
        .as_str();

    let mort_lf = mort_df
        .clone()
        .lazy()
        .select([col("age"), col(mort_col_name).alias("mort_rate")]);

    let result = lf
        .left_join(mort_lf, col("age"), col("age"))
        .with_column(col("mort_rate").fill_null(lit(0.0)).alias("mort_rate"));

    Ok(result.collect()?.lazy()) // To avoid nested lazyframe
}

// Map lapse, Inflation, Expenses and Spot rate assumption
fn __map_other_assumption(lf: LazyFrame, lookup_df: &DataFrame) -> PolarsResult<LazyFrame> {
    let col_name = lookup_df.get_column_names()[1].as_str();

    let lookup_lf = lookup_df
        .clone()
        .lazy()
        .with_column((col("year") - lit(1)).alias("duration")) // Adjust year to duration
        .select([col("duration"), col(col_name)]); // Drop "year" column

    let result = lf
        .left_join(lookup_lf, col("duration"), col("duration"))
        .with_column(col(col_name).fill_null(lit(0.0)).alias(col_name)); // Fill null with 0.0

    Ok(result.collect()?.lazy()) // To avoid nested lazyframe
}

// Discount factor from spot rate
fn __discount_factor(lf: LazyFrame) -> PolarsResult<LazyFrame> {
    let result = lf
        .with_column(
            // Spot rate monthly
            ((lit(1.0) + col("spot_rate")).pow(1.0 / 12.0) - lit(1.0)).alias("spot_rate_mth"),
        )
        .with_column(
            // Discount factor
            (lit(1.0) / (lit(1.0) + col("spot_rate_mth")).pow(col("t").cast(DataType::Float64)))
                .alias("discount_factor"),
        );

    Ok(result.collect()?.lazy())
}

// Expense per policy
fn __exp_pp(lf: LazyFrame) -> PolarsResult<LazyFrame> {
    let lf = lf
        .with_columns(vec![
            // Total real expense per policy
            (col("real_acq_exp_pp") + col("real_mtn_exp_pp")).alias("real_exp_pp"),
            // Inflation factor - for flat curve only
            (lit(1.0) + col("inf_rate"))
                .pow(col("t").cast(DataType::Float64) / lit(12.0))
                .alias("inf_factor"),
        ])
        .with_column(
            // Adjusted expense per policy
            (col("real_exp_pp") * col("inf_factor")).alias("exp_pp"),
        );

    Ok(lf.collect()?.lazy()) // To avoid nested lazyframe
}

// Count policy inforce and decrements
fn __policies_count(lf: LazyFrame, policy_count: f64, term: i32) -> PolarsResult<LazyFrame> {
    let lf = lf.with_columns(vec![
        // Monthly decrement rate
        (lit(1.0) - (lit(1.0) - col("mort_rate")).pow(1.0 / 12.0)).alias("mort_rate_mth"),
        (lit(1.0) - (lit(1.0) - col("lapse_rate")).pow(1.0 / 12.0)).alias("lapse_rate_mth"),
    ]);

    let df = lf.clone().collect()?;

    // Height of the dataframe
    let height = df.height() as usize;

    // Monthly mortality and lapse rate
    let mort_rate_mth = df.column("mort_rate_mth")?.f64()?.to_vec();
    let lapse_rate_mth = df.column("lapse_rate_mth")?.f64()?.to_vec();

    // Create a vector of 0.0 with length equal to lf.height()
    let mut pols_if = Array1::<f64>::zeros(height).to_vec();
    pols_if[0] = policy_count; // Set first element to policy_count

    let mut pols_maturity = Array1::<f64>::zeros(height).to_vec();
    let mut pols_death = Array1::<f64>::zeros(height).to_vec();
    let mut pols_lapse = Array1::<f64>::zeros(height).to_vec();

    for i in 0..(height - 1) {
        if i == 0 {
            pols_if[i] = policy_count;
        } else {
            pols_if[i] =
                pols_if[i - 1] - pols_maturity[i - 1] - pols_death[i - 1] - pols_lapse[i - 1];
        }

        pols_maturity[i] = if i == (term * 12) as usize {
            pols_if[i] // Maturity at the end of the term
        } else {
            0.0 // No maturity before term ends
        };

        pols_death[i] = (pols_if[i] - pols_maturity[i]) * mort_rate_mth[i].unwrap_or(0.0);
        pols_lapse[i] =
            (pols_if[i] - pols_maturity[i] - pols_death[i]) * lapse_rate_mth[i].unwrap_or(0.0);
    }

    // Create a DataFrame from these vectors
    let new_df = df![
        "pols_if" => pols_if,
        "pols_maturity" => pols_maturity,
        "pols_death" => pols_death,
        "pols_lapse" => pols_lapse,
    ]?;

    // Horizontally concatenate the new columns to the existing LazyFrame
    let result = df
        .hstack(&[
            new_df.column("pols_if")?.clone(),
            new_df.column("pols_maturity")?.clone(),
            new_df.column("pols_death")?.clone(),
            new_df.column("pols_lapse")?.clone(),
        ])?
        .lazy();

    Ok(result.collect()?.lazy()) // To avoid nested lazyframe
}

// Net premium calculation - do not consume dataframe
fn ___calculate_net_premium(lf: LazyFrame, sum_insured: f64) -> PolarsResult<f64> {
    // Calculate both PV claims and premium annuities in a single operation
    let result = lf
        .with_columns(vec![
            // Claim per policy
            lit(sum_insured).alias("claim_pp"),
            // Portfolio claims
            (lit(sum_insured) * col("pols_death")).alias("claims"),
        ])
        .with_columns(vec![
            // PV of claims and PV of premium annuities
            (col("claims") * col("discount_factor")).alias("pv_claims_component"),
            (col("pols_if") * col("discount_factor")).alias("pv_annuities_component"),
        ])
        .select([
            col("pv_claims_component").sum().alias("pv_claims"),
            col("pv_annuities_component").sum().alias("prem_annuities"),
        ])
        .collect()?;

    // Extract both values in one operation
    let pv_claims = result
        .column("pv_claims")?
        .get(0)?
        .extract::<f64>()
        .unwrap_or(0.0);

    let prem_annuities = result
        .column("prem_annuities")?
        .get(0)?
        .extract::<f64>()
        .unwrap_or(0.0);

    // Calculate net premium
    let net_premium = if prem_annuities != 0.0 {
        pv_claims / prem_annuities
    } else {
        0.0
    };

    Ok(net_premium)
}

// Complete projection
fn __complete_projection(lf: LazyFrame, sum_insured: f64) -> PolarsResult<LazyFrame> {
    // Calculate net premium
    let net_prem = ___calculate_net_premium(lf.clone(), sum_insured)?;

    // Add net premium to the lazyframe
    let lf = lf
        .with_columns(vec![
            // Claim per policy
            lit(sum_insured).alias("claim_pp"),
            // Loaded premium and round to 2 decimal places
            ((lit(1.0) + col("load_rate")) * lit(net_prem)).alias("prem_pp"),
            // Portfolio expense
            (col("exp_pp") * col("pols_if")).alias("expenses"),
        ])
        .with_columns(vec![
            // Portfolio claims
            (col("claim_pp") * col("pols_death")).alias("claims"),
            // Portfolio premiums
            (col("prem_pp") * col("pols_if")).alias("premiums"),
        ])
        .with_column(
            // Features is simple - Comission is 100% of premium in the first year
            when(col("duration").eq(0))
                .then(col("premiums"))
                .otherwise(lit(0.0))
                .alias("commissions"),
        )
        .with_column(
            (col("premiums") - col("expenses") - col("claims") - col("commissions"))
                .alias("net_cf"),
        );

    Ok(lf.collect()?.lazy()) // To avoid nested lazyframe
}

fn _project_single_model_point(
    mp: &ModelPoint,
    assumptions: &AssumptionSet,
) -> PolarsResult<DataFrame> {
    // Initialize projection dataframe - using all interger values
    let lf = __initialize_lf(mp.id, mp.term, mp.entry_age, mp.sum_insured)?;
    // Map assumptions
    let lf = __map_mort_assumption(lf, &assumptions.mort, &mp.gender)?; // Mortality assumption based on gender
    let lf = __map_other_assumption(lf, &assumptions.lapse)?; // Lapse assumption
    let lf = __map_other_assumption(lf, &assumptions.acq)?; // Acquisition expenses
    let lf = __map_other_assumption(lf, &assumptions.mtn)?; // Maintenance expenses
    let lf = __map_other_assumption(lf, &assumptions.inf)?; // Inflation assumption
    let lf = __map_other_assumption(lf, &assumptions.spot)?; // Spot rate assumption
    let lf = __map_other_assumption(lf, &assumptions.load)?; // Load rate assumption
    // Perform projection
    let lf = __discount_factor(lf)?;
    let lf = __exp_pp(lf)?;
    let lf = __policies_count(lf, mp.policy_count, mp.term)?;
    let lf = __complete_projection(lf, mp.sum_insured)?;

    Ok(lf.collect()?)
}

//---------------------------------------------------------------------------------------------------------
// PUBLIC
//---------------------------------------------------------------------------------------------------------

/*
Using the below command to run the code in parallel with limited threads finish run in 90s vs 400s in non parallel mode
The test is not exhaustive, but it shows that parallel processing can significantly speed up the projection of model points.
$env:RAYON_NUM_THREADS = 8; $env:RUST_MIN_STACK = 33554432; cargo run
*/

//----------------------------------------
// Non-parallel version of the projection
//----------------------------------------
pub fn project_single_run(run_setup: &RunSetup) -> PolarsResult<RunResult> {
    // Convert model points DataFrame to vector
    let model_points_vec = convert_model_points_df_to_vector(&run_setup.model_points_df)?;

    let mut lfs = Vec::with_capacity(model_points_vec.len());

    for mp in &model_points_vec {
        let df = _project_single_model_point(mp, &run_setup.assumptions)?;
        lfs.push(df.lazy());
    }

    // Concatenate all LazyFrames and collect to DataFrame
    let lf = concat(lfs, Default::default())?;
    let final_df = lf.collect()?;

    // Return the result with run setup and projected DataFrame
    let result = RunResult {
        run_setup: run_setup.clone(),
        projected_df: final_df,
    };

    Ok(result)
}

pub fn project_multiple_run(run_setups: &Vec<RunSetup>) -> PolarsResult<MultipleRunResult> {
    let mut results: Vec<RunResult> = Vec::with_capacity(run_setups.len()); // To collect run results

    for i in 0..run_setups.len() {
        let setup = run_setups.get(i).unwrap();
        let result = project_single_run(setup)?;
        results.push(result);
    }

    Ok(MultipleRunResult {
        run_setups: run_setups.clone(),
        run_results: results,
    })
}

//----------------------------------------
// Parallel version of the projection
//----------------------------------------
// Process data in chunks to avoid stack overflow
const CHUNK_SIZE: usize = 100;

pub fn project_single_run_parallel(run_setup: &RunSetup) -> PolarsResult<RunResult> {
    // Convert model points DataFrame to vector
    let model_points_vec = convert_model_points_df_to_vector(&run_setup.model_points_df)?;

    // Process chunks of model points in parallel with limited threads
    let chunks: Vec<&[ModelPoint]> = model_points_vec.chunks(CHUNK_SIZE).collect();

    let chunk_dfs: PolarsResult<Vec<DataFrame>> = chunks
        .into_par_iter()
        .map(|chunk| {
            // Process each chunk sequentially (no nested parallelism)
            let mut lfs = Vec::new();
            for mp in chunk {
                let df = _project_single_model_point(mp, &run_setup.assumptions)?;
                lfs.push(df.lazy());
            }

            // Concatenate LazyFrames within the chunk and collect to DataFrame
            let lf = concat(lfs, Default::default())?;
            lf.collect()
        })
        .collect();

    // Unwrap the result or return the error
    let chunk_dfs = chunk_dfs?;

    // Concatenate all chunk DataFrames
    let final_df = concat(
        chunk_dfs
            .into_iter()
            .map(|df| df.lazy())
            .collect::<Vec<_>>(),
        Default::default(),
    )?
    .collect()?;

    // Return the result with run setup and projected DataFrame
    let result = RunResult {
        run_setup: run_setup.clone(),
        projected_df: final_df,
    };

    Ok(result)
}

pub fn project_multiple_run_parallel(
    run_setups: &Vec<RunSetup>,
) -> PolarsResult<MultipleRunResult> {
    let mut results: Vec<RunResult> = Vec::with_capacity(run_setups.len()); // To collect run results

    for i in 0..run_setups.len() {
        let setup = run_setups.get(i).unwrap();
        let result = project_single_run_parallel(setup)?;
        results.push(result);
    }

    Ok(MultipleRunResult {
        run_setups: run_setups.clone(),
        run_results: results,
    })
}

pub fn project_multiple_run_parallel_x2(
    run_setups: &Vec<RunSetup>,
) -> PolarsResult<MultipleRunResult> {
    let results: Vec<RunResult> = run_setups
        .par_iter()
        .map(|setup| project_single_run_parallel(setup))
        .collect::<PolarsResult<Vec<_>>>()?;

    Ok(MultipleRunResult {
        run_setups: run_setups.clone(),
        run_results: results,
    })
}

### 3.4 Premium rate queries

In [8]:
//---------------------------------------------------------------------------------------------------------
// PUBLIC
//---------------------------------------------------------------------------------------------------------
pub fn get_avg_prem_rate_by_age_and_term(run_result: &RunResult) -> PolarsResult<DataFrame> {
    let proj_df = run_result.projected_df.clone();

    // Compute rate column and aggregate
    let df = proj_df
        .lazy()
        .with_column((col("prem_pp") / col("sum_insured") * lit(1000.0)).alias("rate"))
        .filter(col("t").eq(lit(0)))
        .group_by([col("age"), col("term")])
        .agg([col("rate").mean().alias("avg_rate")])
        .sort(["term", "age"], Default::default())
        .collect()?;

    Ok(df)
}


## 4. Demonstration

### 4.1 Model point generation

- Assume 10k of model points are required. Large number of model point is to proved the efficency over traditional spreadsheet.
- A seed number is to ensure consistency when replication is required.

In [9]:
let model_points_df = generate_model_points(10_000, 987654321)?;
model_points_df

shape: (10_000, 6)
┌───────┬───────────┬────────┬──────┬──────────────┬─────────────┐
│ id    ┆ entry_age ┆ gender ┆ term ┆ policy_count ┆ sum_insured │
│ ---   ┆ ---       ┆ ---    ┆ ---  ┆ ---          ┆ ---         │
│ i32   ┆ i32       ┆ str    ┆ i32  ┆ f64          ┆ f64         │
╞═══════╪═══════════╪════════╪══════╪══════════════╪═════════════╡
│ 1     ┆ 26        ┆ F      ┆ 15   ┆ 1.0          ┆ 994000.0    │
│ 2     ┆ 35        ┆ F      ┆ 10   ┆ 1.0          ┆ 204000.0    │
│ 3     ┆ 48        ┆ F      ┆ 10   ┆ 1.0          ┆ 888000.0    │
│ 4     ┆ 30        ┆ F      ┆ 20   ┆ 1.0          ┆ 789000.0    │
│ 5     ┆ 52        ┆ M      ┆ 20   ┆ 1.0          ┆ 642000.0    │
│ …     ┆ …         ┆ …      ┆ …    ┆ …            ┆ …           │
│ 9996  ┆ 27        ┆ F      ┆ 10   ┆ 1.0          ┆ 146000.0    │
│ 9997  ┆ 58        ┆ F      ┆ 10   ┆ 1.0          ┆ 825000.0    │
│ 9998  ┆ 55        ┆ F      ┆ 20   ┆ 1.0          ┆ 620000.0    │
│ 9999  ┆ 34        ┆ M      ┆ 15   ┆ 1.0  

### 4.2 Assumptions
- The demo only shows one set of assumptions. In practise, we might have to iterate over different assumption sets.
- The notebook tries to show how it can be achieved with a proper setup via Rust Polars crate. The actual implementation might be difference.

In [10]:
// Collect the dataframes into AssumptionSet that can be used later
let assumption_set_01 = AssumptionSet {
    mort: get_mort_df("cso80")?,
    lapse: get_lapse_df("lapse_01")?,
    inf: get_inf_rate_df("inf_01")?,
    acq: get_acq_exp_df("acq_exp_01")?,
    mtn: get_mtn_exp_df("mtn_exp_01")?,
    spot: get_spot_rate_df("spot_rate_01")?,
    load: get_load_rate_df("load_01")?,
};

assumption_set_01

AssumptionSet { mort: shape: (121, 3)
┌─────┬─────────┬─────────┐
│ age ┆ cso80_m ┆ cso80_f │
│ --- ┆ ---     ┆ ---     │
│ i64 ┆ f64     ┆ f64     │
╞═════╪═════════╪═════════╡
│ 0   ┆ 0.00263 ┆ 0.00188 │
│ 1   ┆ 0.00103 ┆ 0.00084 │
│ 2   ┆ 0.00099 ┆ 0.0008  │
│ 3   ┆ 0.00097 ┆ 0.00078 │
│ 4   ┆ 0.00093 ┆ 0.00077 │
│ …   ┆ …       ┆ …       │
│ 116 ┆ 1.0     ┆ 1.0     │
│ 117 ┆ 1.0     ┆ 1.0     │
│ 118 ┆ 1.0     ┆ 1.0     │
│ 119 ┆ 1.0     ┆ 1.0     │
│ 120 ┆ 1.0     ┆ 1.0     │
└─────┴─────────┴─────────┘, lapse: shape: (20, 2)
┌──────┬────────────┐
│ year ┆ lapse_rate │
│ ---  ┆ ---        │
│ i64  ┆ f64        │
╞══════╪════════════╡
│ 1    ┆ 0.4        │
│ 2    ┆ 0.35       │
│ 3    ┆ 0.3        │
│ 4    ┆ 0.25       │
│ 5    ┆ 0.2        │
│ …    ┆ …          │
│ 16   ┆ 0.0        │
│ 17   ┆ 0.0        │
│ 18   ┆ 0.0        │
│ 19   ┆ 0.0        │
│ 20   ┆ 0.0        │
└──────┴────────────┘, inf: shape: (20, 2)
┌──────┬──────────┐
│ year ┆ inf_rate │
│ ---  ┆ ---      │
│ i64  ┆

In [11]:
let assumption_set_02 = AssumptionSet {
    mort: get_mort_df("cso80")?,
    lapse: get_lapse_df("lapse_02")?,
    inf: get_inf_rate_df("inf_02")?,
    acq: get_acq_exp_df("acq_exp_02")?,
    mtn: get_mtn_exp_df("mtn_exp_02")?,
    spot: get_spot_rate_df("spot_rate_01")?,
    load: get_load_rate_df("load_02")?,
};

assumption_set_02

AssumptionSet { mort: shape: (121, 3)
┌─────┬─────────┬─────────┐
│ age ┆ cso80_m ┆ cso80_f │
│ --- ┆ ---     ┆ ---     │
│ i64 ┆ f64     ┆ f64     │
╞═════╪═════════╪═════════╡
│ 0   ┆ 0.00263 ┆ 0.00188 │
│ 1   ┆ 0.00103 ┆ 0.00084 │
│ 2   ┆ 0.00099 ┆ 0.0008  │
│ 3   ┆ 0.00097 ┆ 0.00078 │
│ 4   ┆ 0.00093 ┆ 0.00077 │
│ …   ┆ …       ┆ …       │
│ 116 ┆ 1.0     ┆ 1.0     │
│ 117 ┆ 1.0     ┆ 1.0     │
│ 118 ┆ 1.0     ┆ 1.0     │
│ 119 ┆ 1.0     ┆ 1.0     │
│ 120 ┆ 1.0     ┆ 1.0     │
└─────┴─────────┴─────────┘, lapse: shape: (20, 2)
┌──────┬────────────┐
│ year ┆ lapse_rate │
│ ---  ┆ ---        │
│ i64  ┆ f64        │
╞══════╪════════════╡
│ 1    ┆ 0.15       │
│ 2    ┆ 0.12       │
│ 3    ┆ 0.1        │
│ 4    ┆ 0.08       │
│ 5    ┆ 0.06       │
│ …    ┆ …          │
│ 16   ┆ 0.0        │
│ 17   ┆ 0.0        │
│ 18   ┆ 0.0        │
│ 19   ┆ 0.0        │
│ 20   ┆ 0.0        │
└──────┴────────────┘, inf: shape: (20, 2)
┌──────┬──────────┐
│ year ┆ inf_rate │
│ ---  ┆ ---      │
│ i64  ┆

### 4.3 Run setup

In [12]:
let run_setup_01 = RunSetup {
    description: "Run setup 01".to_string(),
    model_points_df: model_points_df.clone(),
    assumptions: assumption_set_01,
};

run_setup_01

RunSetup { description: "Run setup 01", model_points_df: shape: (10_000, 6)
┌───────┬───────────┬────────┬──────┬──────────────┬─────────────┐
│ id    ┆ entry_age ┆ gender ┆ term ┆ policy_count ┆ sum_insured │
│ ---   ┆ ---       ┆ ---    ┆ ---  ┆ ---          ┆ ---         │
│ i32   ┆ i32       ┆ str    ┆ i32  ┆ f64          ┆ f64         │
╞═══════╪═══════════╪════════╪══════╪══════════════╪═════════════╡
│ 1     ┆ 26        ┆ F      ┆ 15   ┆ 1.0          ┆ 994000.0    │
│ 2     ┆ 35        ┆ F      ┆ 10   ┆ 1.0          ┆ 204000.0    │
│ 3     ┆ 48        ┆ F      ┆ 10   ┆ 1.0          ┆ 888000.0    │
│ 4     ┆ 30        ┆ F      ┆ 20   ┆ 1.0          ┆ 789000.0    │
│ 5     ┆ 52        ┆ M      ┆ 20   ┆ 1.0          ┆ 642000.0    │
│ …     ┆ …         ┆ …      ┆ …    ┆ …            ┆ …           │
│ 9996  ┆ 27        ┆ F      ┆ 10   ┆ 1.0          ┆ 146000.0    │
│ 9997  ┆ 58        ┆ F      ┆ 10   ┆ 1.0          ┆ 825000.0    │
│ 9998  ┆ 55        ┆ F      ┆ 20   ┆ 1.0          ┆ 

In [13]:
let run_setup_02 = RunSetup {
    description: "Run setup 02".to_string(),
    model_points_df: model_points_df.clone(),
    assumptions: assumption_set_02,
};

run_setup_02

RunSetup { description: "Run setup 02", model_points_df: shape: (10_000, 6)
┌───────┬───────────┬────────┬──────┬──────────────┬─────────────┐
│ id    ┆ entry_age ┆ gender ┆ term ┆ policy_count ┆ sum_insured │
│ ---   ┆ ---       ┆ ---    ┆ ---  ┆ ---          ┆ ---         │
│ i32   ┆ i32       ┆ str    ┆ i32  ┆ f64          ┆ f64         │
╞═══════╪═══════════╪════════╪══════╪══════════════╪═════════════╡
│ 1     ┆ 26        ┆ F      ┆ 15   ┆ 1.0          ┆ 994000.0    │
│ 2     ┆ 35        ┆ F      ┆ 10   ┆ 1.0          ┆ 204000.0    │
│ 3     ┆ 48        ┆ F      ┆ 10   ┆ 1.0          ┆ 888000.0    │
│ 4     ┆ 30        ┆ F      ┆ 20   ┆ 1.0          ┆ 789000.0    │
│ 5     ┆ 52        ┆ M      ┆ 20   ┆ 1.0          ┆ 642000.0    │
│ …     ┆ …         ┆ …      ┆ …    ┆ …            ┆ …           │
│ 9996  ┆ 27        ┆ F      ┆ 10   ┆ 1.0          ┆ 146000.0    │
│ 9997  ┆ 58        ┆ F      ┆ 10   ┆ 1.0          ┆ 825000.0    │
│ 9998  ┆ 55        ┆ F      ┆ 20   ┆ 1.0          ┆ 

### 4.3 Cashflow projection and net premium calculation
PV Claims = PV Gross prem - PV Commission - PV Expense = PV of Net prem = Net prem * prem annuities.

Hence Net prem = PV Claims / Prem annuities

In [14]:
let run_setups = vec![run_setup_01, run_setup_02];
let multi_run_results = project_multiple_run_parallel_x2(&run_setups)?;
// Aggregated results of multiple runs
let multi_run_proj_df = multi_run_results.aggregate_projected_df()?;

multi_run_proj_df

shape: (3_618_680, 33)
┌───────┬──────┬─────────────┬─────┬───┬─────────────┬─────────────┬────────┬─────────────────┐
│ id    ┆ term ┆ sum_insured ┆ t   ┆ … ┆ commissions ┆ net_cf      ┆ run_id ┆ run_description │
│ ---   ┆ ---  ┆ ---         ┆ --- ┆   ┆ ---         ┆ ---         ┆ ---    ┆ ---             │
│ i32   ┆ i32  ┆ f64         ┆ i32 ┆   ┆ f64         ┆ f64         ┆ i32    ┆ str             │
╞═══════╪══════╪═════════════╪═════╪═══╪═════════════╪═════════════╪════════╪═════════════════╡
│ 1     ┆ 15   ┆ 994000.0    ┆ 0   ┆ … ┆ 178.794982  ┆ -459.454712 ┆ 0      ┆ Run setup 01    │
│ 1     ┆ 15   ┆ 994000.0    ┆ 1   ┆ … ┆ 171.326473  ┆ -440.832399 ┆ 0      ┆ Run setup 01    │
│ 1     ┆ 15   ┆ 994000.0    ┆ 2   ┆ … ┆ 164.169934  ┆ -422.965068 ┆ 0      ┆ Run setup 01    │
│ 1     ┆ 15   ┆ 994000.0    ┆ 3   ┆ … ┆ 157.312333  ┆ -405.822103 ┆ 0      ┆ Run setup 01    │
│ 1     ┆ 15   ┆ 994000.0    ┆ 4   ┆ … ┆ 150.741184  ┆ -389.374129 ┆ 0      ┆ Run setup 01    │
│ …     ┆ …    ┆ 

In [16]:
// Premium rates - assume that run 01 is used to obtain premium rates
let run_result_01 = multi_run_results.run_results[0].clone();
let prem_rate_df = get_avg_prem_rate_by_age_and_term(&run_result_01)?;

prem_rate_df

not found: prem_pp

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
DF ["id", "term", "sum_insured", "t", ...]; PROJECT */31 COLUMNS


### 4.4 Jupyter notebook limitation with Rust Kernel
There is a limitation with lazyframe in Jupyter notebook hence a full demonstration cannot be shown here. However, refer to queries.rs for full solution to obtain the premium rate. (eg fn get_avg_prem_by_age_and_term)

Note: Try to run the last cell will result in unexplainable error.

The pricing is basically completed with several more extra due dilligence steps including smoothing the curve premium rate. We shall use spline technique in order to do so.

Another solution is that some actuary might handle smooth manually with interpolation. However I do not see the elegancy and efficency in this approach as it would be hard to replicate over different procedures.